# SteamCharts source refresh pilot

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

![06 / Source Refresh Pilot — Newer observations, overlap agreement and collection limits](assets/pixel_theme/06_source_refresh.svg)

At this stage, newer monthly observations were needed before assessing a 2027 forecast. This pilot checks whether published SteamCharts history can extend the existing snapshot while preserving comparable measurements and a record of unavailable data.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from notebook_display import configure_notebook, present, report_table
configure_notebook()

RUN_ID = "20261025T085419Z_52879d"
REPORT = ROOT / "reports" / "steamcharts_pilot" / RUN_ID
summary = json.loads((REPORT / "summary.json").read_text(encoding="utf-8"))
audit = pd.read_csv(REPORT / "app_audit.csv")
monthly = pd.read_csv(REPORT / "monthly_players.csv", parse_dates=["month"])
overlap = pd.read_csv(REPORT / "overlap_comparison.csv")
report_table(pd.Series(summary, name="Pilot result").to_frame(), 'Pilot snapshot and scope')


,Pilot result
Run id,20261025T085419Z_52879d
As of,2026-10-25
Baseline cutoff,Sep 2025
Latest complete calendar month,Sep 2026
Selected apps,20
Successful apps,19
Failed apps,1
Completed month rows,"2,330"
Overlap rows,"1,885"
Overlap mismatches,0


## 1. Check retrieval and recent coverage

Twenty apps were selected to include large audiences, small audiences, incomplete histories, unmatched metadata and software holdouts. This deliberate sample probes different failure cases; its success rate cannot estimate coverage across Steam.

In [2]:
report_table(audit[["app_id", "name", "status", "rows", "last_month", "has_latest_complete_month", "error"]], 'Retrieval status and latest available month')


App ID,Name,Status,Rows,Last month,Latest complete month available,Error
730,Counter-Strike 2,ok,171,Sep 2026,True,None recorded
252490,Rust,ok,154,Sep 2026,True,None recorded
289070,Sid Meier’s Civilization® VI,ok,120,Sep 2026,True,None recorded
570,Dota 2,ok,171,Sep 2026,True,None recorded
578080,PUBG: BATTLEGROUNDS,ok,115,Sep 2026,True,None recorded
413150,Stardew Valley,ok,130,Sep 2026,True,None recorded
394360,Hearts of Iron IV,ok,126,Sep 2026,True,None recorded
242050,Assassin’s Creed® IV Black Flag™,ok,154,Sep 2026,True,None recorded
436150,Governor of Poker 3,ok,128,Sep 2026,True,None recorded
431960,Wallpaper Engine,ok,121,Sep 2026,True,None recorded


Nineteen pages produced usable history, while Potions: A Curious Tale returned HTTP 500. Seventeen apps have a September 2026 observation; Sentou Gakuen: Revival ends in February and HellSign ends in July. Their later activity remains unknown, and the failed request provides no evidence about player counts.

## 2. Compare the old and refreshed measurements

Matching app IDs and months allows a direct comparison of the same observations. A tolerance of 0.011 players accommodates values rounded to two decimal places; discrepancies remain visible in the saved comparison.

In [3]:
assert not monthly.duplicated(["app_id", "month"]).any()
assert monthly["month"].max() < pd.Timestamp(summary["as_of"]).to_period("M").to_timestamp()
assert summary["overlap_mismatches"] == int((~overlap["within_rounding_tolerance"]).sum())
report_table(audit[["name", "overlap_months", "overlap_mismatches", "new_months", "internal_missing_months"]], 'Overlap agreement and history gaps')


Name,Overlap months,Overlap mismatches,New months,Internal missing months
Counter-Strike 2,159,0,12,0
Rust,142,0,12,0
Sid Meier’s Civilization® VI,108,0,12,0
Dota 2,0,0,12,0
PUBG: BATTLEGROUNDS,103,0,12,0
Stardew Valley,118,0,12,0
Hearts of Iron IV,114,0,12,0
Assassin’s Creed® IV Black Flag™,142,0,12,0
Governor of Poker 3,116,0,12,0
Wallpaper Engine,109,0,12,0


All 1,885 overlapping observations match within tolerance. The pilot contains 215 observations after the old September 2025 cutoff, alongside historical observations for apps absent from that player snapshot. These counts describe different checks: agreement establishes comparability where both sources have records, while newly recovered history broadens the information available for review.

## 3. Review catalog and eligibility changes

Dota 2 has no overlapping records in the existing player snapshot despite its large audience, which makes catalog selection important for future category totals. IdleOn has a November 2025 metadata release date, while Hades II is dated September 25, 2025, within the original snapshot's final month. The first pilot filter used September 1 as its boundary and incorrectly grouped Hades II with post-cutoff releases; future runs now compare calendar months. Both apps have earlier SteamCharts activity, so metadata release dates need interpretation before they define inclusion.

In [4]:
report_table(audit.loc[audit["overlap_months"].eq(0) | audit["status"].ne("ok") | ~audit["has_latest_complete_month"], ["app_id", "name", "selection_reason", "first_month", "last_month", "overlap_months", "error"]], 'Apps requiring a coverage or eligibility review')


,App ID,Name,Selection reason,First month,Last month,Overlap months,Error
Group,,,,,,,
3,570,Dota 2,Large-title coverage probe,Jul 2012,Sep 2026,0,None recorded
11,405680,Sentou Gakuen: Revival,History with many internal gaps,Apr 2016,Feb 2026,9,None recorded
12,378690,Potions: A Curious Tale,History with many internal gaps,Unavailable,Unavailable,0,HTTP 500; no history inferred
13,606500,HellSign,Small-audience observed app,Nov 2018,Jul 2026,80,None recorded
15,1476970,IdleOn,Metadata release date after old player cutoff,Apr 2021,Sep 2026,0,None recorded
16,1145350,Hades II,Metadata release date after old player cutoff,May 2024,Sep 2026,0,None recorded


Wallpaper Engine and Soundpad are retrieval probes that remain subject to the existing eligibility policy. Fetching their history does not add them to category model inputs. Likewise, missing metadata still requires a separate metadata decision before category attribution.

## Pilot findings

![Checkpoint 5 — findings, interpretation and the next decision](assets/pixel_theme/checkpoint_05.svg)

The pilot provided evidence for a carefully controlled source refresh: overlapping measurements agree, and newer completed months are available for most selected apps. Scaling would have required an access plan, a defined app catalog and explicit treatment of gaps, failed requests and changing labels. The original database and baseline evaluation retain their September 2025 cutoff.

The code separates retrieval, parsing and audit outputs so each stage can be inspected. Cached HTML and SHA-256 hashes preserve what was retrieved, while offline replay rebuilds the report without another network request. The pilot later became a frozen v2 input; no further collection is planned. See the [pilot manifest](../reports/steamcharts_pilot/20261025T085419Z_52879d/source_manifest.json) for recorded retrieval evidence.